# Financial Fraud Detection - Exploratory Data Analysis

This notebook performs initial exploration of the PaySim financial transactions dataset to understand fraud patterns and data characteristics.

## Load Dataset

Load the PaySim transaction dataset into a Pandas DataFrame for analysis.

In [3]:
import pyspark
print(pyspark.__version__)

3.5.9


In [4]:
import sys
import pyspark

print("Python:", sys.version)
print("Executable:", sys.executable)
print("PySpark:", pyspark.__version__)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Executable: c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\.venv311\Scripts\python.exe
PySpark: 3.5.9


In [5]:
import sys
print(sys.executable)

c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\.venv311\Scripts\python.exe


In [6]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot


In [7]:
import pandas as pd

df = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")

## Preview Dataset

Display the first few records to understand the structure and available features.

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FraudDetectionPipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.9


In [9]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

print("Spark EDA libraries imported")

Spark EDA libraries imported


In [10]:
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
csv_path = data_dir / "PS_20174392719_1491204439457_log.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"CSV file not found: {csv_path}")

fraud_df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully")
print("CSV path:", csv_path)

Dataset loaded successfully
CSV path: c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\data\PS_20174392719_1491204439457_log.csv


In [11]:
fraud_df.show(5)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

# Check Dataset Shape

In [12]:
print("Rows:", fraud_df.count())
print("Columns:", len(fraud_df.columns))

Rows: 6362620
Columns: 11


# Check Schema

In [13]:
fraud_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



## 1. Dataset Overview

This section provides a basic understanding of the financial transaction dataset.
We will analyze the number of records, available features, and data distribution
using PySpark DataFrame operations.

In [14]:
# Total records and columns

print("Total Transactions:", fraud_df.count())
print("Total Features:", len(fraud_df.columns))

Total Transactions: 6362620
Total Features: 11


## 2. Sample Transaction Records

Displaying sample transactions to understand the structure and values of the dataset.

In [15]:
fraud_df.show(10)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

## 3. Missing Value Analysis

Checking missing values in each column to ensure data quality before further analysis.

In [16]:
from pyspark.sql.functions import col, sum

missing_values = fraud_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in fraud_df.columns
    ]
)

missing_values.show()

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



## 4. Transaction Type Analysis

Analyzing different transaction categories to understand customer transaction behavior.

In [17]:
transaction_type = fraud_df.groupBy("type") \
    .count() \
    .orderBy(col("count").desc())

transaction_type.show()

+--------+-------+
|    type|  count|
+--------+-------+
|CASH_OUT|2237500|
| PAYMENT|2151495|
| CASH_IN|1399284|
|TRANSFER| 532909|
|   DEBIT|  41432|
+--------+-------+



## 5. Fraud Transaction Distribution

The target variable `isFraud` indicates whether a transaction is fraudulent.
This analysis helps identify the imbalance between normal and fraudulent transactions.

In [18]:
fraud_distribution = fraud_df.groupBy("isFraud") \
    .count()

fraud_distribution.show()

+-------+-------+
|isFraud|  count|
+-------+-------+
|      0|6354407|
|      1|   8213|
+-------+-------+



## 6. Fraud Percentage

Calculating the percentage of fraudulent transactions in the complete dataset.

In [19]:
total = fraud_df.count()

fraud_count = fraud_df.filter(
    col("isFraud") == 1
).count()

fraud_percentage = (fraud_count / total) * 100

print("Fraud Percentage:", fraud_percentage)

Fraud Percentage: 0.12908204481801522


## 7. Transaction Amount Analysis

Analyzing transaction amounts to identify patterns between fraudulent and normal transactions.

In [20]:
fraud_df.select(
    "amount"
).describe().show()

+-------+------------------+
|summary|            amount|
+-------+------------------+
|  count|           6362620|
|   mean|179861.90354913412|
| stddev| 603858.2314629498|
|    min|               0.0|
|    max|     9.244551664E7|
+-------+------------------+



## 8. Average Amount Comparison

Comparing average transaction amounts between fraudulent and legitimate transactions.

In [21]:
fraud_df.groupBy("isFraud") \
    .avg("amount") \
    .show()

+-------+------------------+
|isFraud|       avg(amount)|
+-------+------------------+
|      0| 178197.0417274114|
|      1|1467967.2991403837|
+-------+------------------+



## 9. Fraud Pattern by Transaction Type

Identifying which transaction types contain higher fraud activity.

In [22]:
fraud_by_type = fraud_df.groupBy("type","isFraud") \
    .count() \
    .orderBy("type")

fraud_by_type.show()

+--------+-------+-------+
|    type|isFraud|  count|
+--------+-------+-------+
| CASH_IN|      0|1399284|
|CASH_OUT|      1|   4116|
|CASH_OUT|      0|2233384|
|   DEBIT|      0|  41432|
| PAYMENT|      0|2151495|
|TRANSFER|      0| 528812|
|TRANSFER|      1|   4097|
+--------+-------+-------+



## 10. Feature Correlation Analysis

Checking relationships between numerical features to identify important variables
for fraud detection modeling.

In [23]:
numeric_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

for column in numeric_columns:
    correlation = fraud_df.stat.corr(column,"isFraud")
    print(column, ":", correlation)

amount : 0.07668842884028321
oldbalanceOrg : 0.010154421850332844
newbalanceOrig : -0.008148161267570215
oldbalanceDest : -0.005885278228051785
newbalanceDest : 0.0005353470683179229


In [24]:
import os

print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

SPARK_HOME: C:\Spark\spark-3.5.9-bin-hadoop3\spark-3.5.9-bin-hadoop3
HADOOP_HOME: C:\hadoop
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot


In [25]:
import os

print("HADOOP_HOME =", os.environ.get("HADOOP_HOME"))

HADOOP_HOME = C:\hadoop


In [26]:
try:
    fraud_df.write.mode("overwrite").parquet(str(output_path))
except Exception as e:
    import traceback
    traceback.print_exc()

Traceback (most recent call last):
  File "C:\Users\Suji\AppData\Local\Temp\ipykernel_18780\1895990182.py", line 2, in <module>
    fraud_df.write.mode("overwrite").parquet(str(output_path))
                                                 ^^^^^^^^^^^
NameError: name 'output_path' is not defined



## 11. Saving Processed Dataset

Saving the Spark DataFrame in Parquet format for faster future processing.

In [27]:
from pathlib import Path
import shutil

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
output_path = data_dir / "fraud_spark_processed"

data_dir.mkdir(parents=True, exist_ok=True)

# Remove any existing output directory to avoid Spark write conflicts
if output_path.exists():
    shutil.rmtree(output_path)

fraud_df.write \
    .mode("overwrite") \
    .parquet(str(output_path))

print("Processed data saved to:", output_path)

Py4JJavaError: An error occurred while calling o144.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(Fi

## Conclusion

The Spark-based EDA identified important characteristics of financial transactions,
including fraud distribution, transaction type patterns, amount behavior, and
important numerical features. These insights will be used for feature engineering
and fraud detection model development.

In [1]:
spark.stop()

# recreate Spark session

fraud_df.write \
    .mode("overwrite") \
    .parquet(str(output_path))

NameError: name 'spark' is not defined